# Module 04 — Inside an NVIDIA T4

This notebook follows two measurable pieces of a simplified 1B-model execution: model-sized FP16 storage and a GPU-resident matrix multiplication. It does not load a complete language model.

## Setup

Run this on a Lightning AI Studio with one NVIDIA T4. The memory experiment allocates about 2 GB, so leave enough VRAM for the runtime.

In [ ]:
import statistics

import torch

assert torch.cuda.is_available(), "Select a CUDA runtime."
device = torch.device("cuda:0")
props = torch.cuda.get_device_properties(device)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", props.name)
print("Compute capability:", f"{props.major}.{props.minor}")
print("VRAM:", round(props.total_memory / 2**30, 2), "GiB")

## Experiment A — Simulate 1B FP16 parameter storage

One billion FP16 elements use one billion × two bytes, or about 2 GB. This is storage simulation only: a real model also needs activations, KV cache, workspaces, and runtime metadata.

In [ ]:
def show_memory(label):
    # Allocated is live tensor storage; reserved includes cached allocator
    # blocks that PyTorch can reuse for later allocations.
    allocated = torch.cuda.memory_allocated(device) / 2**30
    reserved = torch.cuda.memory_reserved(device) / 2**30
    print(f"{label}: allocated={allocated:.3f} GiB, reserved={reserved:.3f} GiB")

parameter_count = 1_000_000_000
bytes_per_parameter = torch.tensor([], dtype=torch.float16).element_size()
show_memory("Before allocation")

# Allocate a flat tensor instead of a model so the memory arithmetic is
# isolated from architecture-specific tensors and initialization logic.
weights = torch.empty(parameter_count, dtype=torch.float16, device=device)
weights.zero_()  # Touch the allocation so the device writes are real.
torch.cuda.synchronize(device)

raw_gib = parameter_count * bytes_per_parameter / 2**30
print(f"Raw tensor size: {raw_gib:.3f} GiB")
show_memory("After allocation")

# Release the simulated weights before the next experiment.
del weights
torch.cuda.empty_cache()
show_memory("After release")

## Experiment B — Watch a large matrix operation execute

The matrices remain on the GPU during timing. CUDA events bracket device execution, and the operation count uses two FLOPs per multiply-add.

In [ ]:
rows = 1_024
hidden = 4_096
repeats = 20
dtype = torch.float16

# A @ B has shapes [1024, 4096] @ [4096, 4096].
A = torch.randn(rows, hidden, dtype=dtype, device=device)
B = torch.randn(hidden, hidden, dtype=dtype, device=device)

# Warm up kernel selection and library setup.
for _ in range(5):
    C = A @ B
torch.cuda.synchronize(device)

samples_ms = []
for _ in range(repeats):
    # Python may return before CUDA finishes; events measure device time.
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    C = A @ B
    end.record()
    end.synchronize()
    samples_ms.append(start.elapsed_time(end))

median_ms = statistics.median(samples_ms)
flops = 2 * rows * hidden * hidden
tflops = flops / (median_ms / 1000) / 1e12
print("A shape:", tuple(A.shape))
print("B shape:", tuple(B.shape))
print("C shape:", tuple(C.shape))
print("Median device time:", round(median_ms, 3), "ms")
print("Approximate throughput:", round(tflops, 2), "TFLOP/s")

## Observe the device

In a separate Lightning AI terminal, run `nvidia-smi -l 1` while the benchmark runs. It can show memory usage, utilization, and power. It does not directly report the percentage of CUDA cores or Tensor Cores executing useful instructions; detailed counters require a profiler.

Interpret the result by separating storage, data movement, and arithmetic. A high utilization reading does not automatically mean peak Tensor Core throughput.